# Nightly SN Ia Supervision Notebook

Interactive review of candidates produced by `run_tonight.py`.

**Workflow:**
1. Set the night directory (e.g., `nights/ut20260301`)
2. Load candidates and light curve data
3. Review fits, plots, merit scores
4. Optionally adjust the observing plan

Run `python run_tonight.py <MJD>` first to generate the night directory.

In [ ]:
# ====================== CONFIGURATION ======================
NIGHT_DIR = '../nights/ut20260301'  # Path to night directory
# ============================================================

import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('nightly_supervision.ipynb'))))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML, Image

%matplotlib inline
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt

night_path = Path(NIGHT_DIR)
assert night_path.exists(), f'Night directory not found: {night_path}'

# Load candidates CSV
csv_path = night_path / 'candidates.csv'
candidates = pd.read_csv(csv_path)
print(f'Night: {night_path.name}')
print(f'Candidates: {len(candidates)}')
print(f'Columns: {list(candidates.columns)}')

## Candidate Summary Table

All candidates sorted by merit score (best first). Key columns:
- **merit**: follow-up priority (higher = better)
- **peak_mag**: fitted peak brightness (AB mag)
- **delta_t**: days since peak (positive = past peak)
- **fit_method**: `villar_mb` (multi-band Villar) or `parabola`
- **surveys**: photometry sources used (Rubin, ZTF, ATLAS)

In [ ]:
# Display summary table
display_cols = ['diaObjectId', 'ra', 'dec', 'ddf_field', 'peak_mag', 'peak_band',
                'delta_t', 'merit', 'sn_score', 'fit_method', 'n_points', 'surveys']
display_cols = [c for c in display_cols if c in candidates.columns]

styled = candidates[display_cols].style.format({
    'ra': '{:.5f}', 'dec': '{:.5f}',
    'peak_mag': '{:.2f}', 'delta_t': '{:+.1f}',
    'merit': '{:.4f}', 'sn_score': '{:.3f}',
}, na_rep='--').background_gradient(subset=['merit'], cmap='YlOrRd')

display(styled)

## Diagnostic Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Merit vs peak magnitude
ax = axes[0, 0]
valid = candidates[candidates['merit'].notna() & (candidates['merit'] > 0)]
ax.scatter(valid['peak_mag'], valid['merit'], c='steelblue', s=30, alpha=0.7, edgecolors='k', linewidth=0.3)
ax.set_xlabel('Peak Magnitude (AB)')
ax.set_ylabel('Merit Score')
ax.set_title(f'Merit vs Brightness ({len(valid)} with merit > 0)')
ax.grid(True, alpha=0.3)

# 2. Merit vs delta_t
ax = axes[0, 1]
ax.scatter(valid['delta_t'], valid['merit'], c='firebrick', s=30, alpha=0.7, edgecolors='k', linewidth=0.3)
ax.set_xlabel('Days Since Peak')
ax.set_ylabel('Merit Score')
ax.set_title('Merit vs Time from Peak')
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax.grid(True, alpha=0.3)

# 3. Sky distribution
ax = axes[1, 0]
sc = ax.scatter(candidates['ra'], candidates['dec'],
                c=candidates['merit'].fillna(0), cmap='YlOrRd',
                s=40, alpha=0.8, edgecolors='gray', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='Merit')
ax.set_xlabel('RA (deg)')
ax.set_ylabel('Dec (deg)')
ax.set_title('Sky Distribution')
ax.grid(True, alpha=0.3)

# 4. SN score vs merit
ax = axes[1, 1]
if 'sn_score' in candidates.columns:
    mask = candidates['sn_score'].notna() & candidates['merit'].notna()
    ax.scatter(candidates.loc[mask, 'sn_score'], candidates.loc[mask, 'merit'],
              c='forestgreen', s=30, alpha=0.7, edgecolors='k', linewidth=0.3)
    ax.set_xlabel('Fink SN Score')
    ax.set_ylabel('Merit Score')
    ax.set_title('Classification vs Follow-up Priority')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print(f'\nMerit > 0: {(candidates["merit"] > 0).sum()} / {len(candidates)}')
print(f'Merit > 0.1: {(candidates["merit"] > 0.1).sum()}')
print(f'Merit > 0.5: {(candidates["merit"] > 0.5).sum()}')
if 'n_ztf' in candidates.columns:
    print(f'With ZTF: {(candidates["n_ztf"] > 0).sum()}')
if 'n_atlas' in candidates.columns:
    print(f'With ATLAS: {(candidates["n_atlas"] > 0).sum()}')

## Light Curve Gallery

Display light curve plots from the night directory, ordered by merit.

In [ ]:
lc_dir = night_path / 'lightcurves'

# Show top candidates by merit
top = candidates.sort_values('merit', ascending=False).head(12)

for _, row in top.iterrows():
    did = str(row['diaObjectId'])
    did_short = did[-12:]
    png = lc_dir / f'{did_short}.png'
    if png.exists():
        merit_str = f'{row["merit"]:.4f}' if pd.notna(row['merit']) else '--'
        mag_str = f'{row["peak_mag"]:.1f}' if pd.notna(row['peak_mag']) else '--'
        dt_str = f'{row["delta_t"]:+.0f}d' if pd.notna(row['delta_t']) else '--'
        ddf = row.get('ddf_field', '') or ''
        surveys = row.get('surveys', 'Rubin') or 'Rubin'
        
        print(f'\n--- {did_short} | merit={merit_str} | mag={mag_str} | dt={dt_str} | {ddf} | {surveys} ---')
        display(Image(filename=str(png), width=800))
    else:
        print(f'\n--- {did_short}: no plot ---')

## Inspect Individual Candidate

Set `IDX` below to the row index from the summary table to examine a single candidate in detail.
This cell fetches a fresh light curve from Fink and re-fits.

In [ ]:
IDX = 0  # Row index from candidates table

from broker_clients.fink_client import FinkLSSTClient
from core.peak_fitting import (
    fit_parabola, fit_villar_multiband, plot_mag, clean_light_curve
)

row = candidates.iloc[IDX]
did = str(row['diaObjectId'])
print(f'Candidate: {did}')
print(f'  RA, Dec: {row["ra"]:.5f}, {row["dec"]:.5f}')
print(f'  DDF: {row.get("ddf_field", "--")}')
print(f'  SN score: {row.get("sn_score", "--")}')
print(f'  Peak: {row.get("peak_band", "?")} = {row.get("peak_mag", "--"):.2f} at MJD {row.get("peak_mjd", "--"):.1f}')
print(f'  delta_t: {row.get("delta_t", "--"):+.1f}d')
print(f'  Merit: {row.get("merit", "--"):.4f}')
print(f'  Fit method: {row.get("fit_method", "--")}')
print(f'  N points: {row.get("n_points", "--")} across {row.get("n_bands", "--")} bands')
print()

# Fetch fresh light curve
fink = FinkLSSTClient()
lc = fink.get_light_curve(did, include_forced=True)
if lc is not None:
    print(f'Fetched {len(lc)} points from Fink')
    lc_clean = clean_light_curve(lc)
    print(f'After cleaning: {len(lc_clean)} points')
    print(f'Bands: {dict(lc_clean["band"].value_counts())}')
    
    # Re-fit
    par = fit_parabola(lc)
    vil = fit_villar_multiband(lc)
    
    fit_result = {'parabola': par, 'villar': vil}
    fig = plot_mag(lc, fit_result, object_id=did, figsize=(14, 7))
    plt.show()
    
    # Per-band results
    print('\nPer-band Villar fits:')
    for band, res in vil.get('per_band', {}).items():
        print(f'  {band}: peak_mag={res["peak_mag"]:.2f} at MJD {res["peak_mjd"]:.1f}'
              f'  chi2/dof={res["chi2_dof"]:.2f}  status={res["status"]}'
              f'  N={res["n_points"]}')
    
    print('\nPer-band parabola fits:')
    for band, res in par.get('per_band', {}).items():
        print(f'  {band}: peak_mag={res["peak_mag"]:.2f} at MJD {res["peak_mjd"]:.1f}'
              f'  chi2/dof={res["chi2_dof"]:.2f}  status={res["status"]}'
              f'  N={res["n_points"]}')
else:
    print('Could not fetch light curve')

## Observing Schedule

The RA-ordered schedule from `run_tonight.py`, with the Magellan catalog file.

In [ ]:
# Display observing schedule
sched_path = night_path / 'observing_schedule.txt'
if sched_path.exists():
    print(sched_path.read_text())
else:
    print('No observing schedule found')

# Catalog file
cat_path = night_path / 'magellan_plan.cat'
if cat_path.exists():
    print(f'\nMagellan catalog: {cat_path}')
    print(f'Copy to TCS: scp {cat_path.resolve()} user@magellan:catalogs/')

## Edit Observing Plan

Remove candidates from the plan, then regenerate the Magellan catalog and schedule.

In [ ]:
# Set REMOVE_IDS to a list of diaObjectId strings to drop from the plan
REMOVE_IDS = []  # e.g., ['170028486211141785', '314007346915311715']

if REMOVE_IDS:
    from core.magellan_planning import write_magellan_catalog, radec_to_sexagesimal
    
    plan = candidates[~candidates['diaObjectId'].astype(str).isin(REMOVE_IDS)].copy()
    plan = plan.sort_values('ra').reset_index(drop=True)
    
    # Regenerate catalog
    cat_path = night_path / 'magellan_plan.cat'
    write_magellan_catalog(plan, str(cat_path))
    print(f'Regenerated catalog: {len(plan)} targets (removed {len(REMOVE_IDS)})')
    
    # Regenerate schedule
    sched_path = night_path / 'observing_schedule_edited.txt'
    lines = [f'# Edited Observing Schedule — {len(plan)} targets']
    lines.append(f'# Removed: {", ".join(REMOVE_IDS)}')
    lines.append('#')
    for i, (_, r) in enumerate(plan.iterrows()):
        ra_s, dec_s = radec_to_sexagesimal(r['ra'], r['dec'])
        did = str(r['diaObjectId'])[-12:]
        pmag = f"{r['peak_mag']:.1f}" if pd.notna(r['peak_mag']) else '--'
        merit = f"{r['merit']:.3f}" if pd.notna(r['merit']) else '--'
        lines.append(f'  {i+1:3d}  {did:20s}  {ra_s}  {dec_s}  mag={pmag}  merit={merit}')
    sched_path.write_text('\n'.join(lines) + '\n')
    print(f'Edited schedule: {sched_path}')
else:
    print('No candidates to remove. Set REMOVE_IDS above to edit the plan.')

## Re-run Pipeline

Convenience cell to re-run `run_tonight.py` from within the notebook.

In [ ]:
# Uncomment and set MJD to re-run
# !python ../run_tonight.py 61100 --max-candidates 100 --no-observability